In [ ]:
#todo I'd like to create a version of this that can have dynamically sized convolutional layers
import numpy as np
from dataclasses import dataclass

print("start")

@dataclass
class NN_Config:
    kernel_shape : tuple
    

class ConvLayer:
    def __init__(self, activations: np.ndarray, z_values: np.ndarray):
        self.activations = activations
        self.z_values = z_values
    
    @classmethod
    def from_shape(cls, layer_size, dtype=np.float32, **kwargs):
        activations = np.zeros(layer_size, dtype=dtype)
        z_vals = np.zeros(layer_size, dtype=dtype)
        return cls(activations, z_vals)

class ConvWeights:
    def __init__(self, weights: np.ndarray):
        self.weights = weights

    @classmethod
    def from_shape(cls, kernel_shape, curr_ly_shape, dtype=np.float32, init=True, **kwargs):
        if init:
        #ly_weights = np.random.rand(prev_ly_size, curr_ly_size)
            ly_weights = np.random.uniform(-1,1, size=(*kernel_shape, *curr_ly_shape))
        else:
            ly_weights = np.zeros(*kernel_shape, *curr_ly_shape)
        return cls( ly_weights)

    def from_dynamic_shape(cls, kernel_shape, prev_ly_shape, curr_ly_shape, dtype=np.float32, init=True, **kwargs):
        print("not yet implemented :(")

class SingleDSimpleLayer:
    def __init__(self, activations, z_values: np.ndarray):
        self.activations = activations
        self.z_values = z_values

    @classmethod
    def create_layer(cls, layer_size, dtype=np.float32, **kwargs):
        activations = np.zeros(layer_size, dtype=dtype)
        z_vals = np.zeros(layer_size, dtype=dtype)        
        return cls(activations, z_vals)

class ConvBias:
    def __init__(self, biases: np.ndarray):
        self.biases = biases

    @classmethod
    def from_shape(cls, curr_ly_shape, dtype=np.float32, init=True, **kwargs):
        if init:
            biases = np.random.rand(*curr_ly_shape)
        else:
            biases = np.zeros(curr_ly_shape)
        return cls(biases)

    def from_dynamic_shape(cls, kernel_shape, prev_ly_shape, curr_ly_shape, dtype=np.float32, init=True, **kwargs):
        print("not yet implemented :(")

class NN:
    def __init__(self, config:NN_Config, layers: list, weights: list, biases: list):
        self.config = config
        self.layers = layers # layers hold both actual activations and z values
        self.weights = weights
        self.biases = biases

    @classmethod
    def create_network(cls, config:NN_Config, in_layer_size: tuple, hidden_layer_num: tuple, hidden_layer_size: tuple, out_layer_size: int, **kwargs):
        layer0 = ConvLayer.from_shape(in_layer_size)
        hidden_lys = []
        for i in range(hidden_layer_num):
            hidden_lys.append(ConvLayer.from_shape(hidden_layer_size))
        layer_fn = SingleDSimpleLayer.create_layer(out_layer_size)
        layers = [layer0, *hidden_lys, layer_fn]

        biases =[]
        for i in range(1, len(layers), 1):
            biases.append(ConvBias.from_shape(np.shape(layers[i].activations), init=False))


        weights = []
        for i in range(1, len(layers), 1):
            weights.append(ConvWeights.from_shape(config.kernel_shape, np.shape(layers[i].activations), init=True))


        return cls(config, layers, weights, biases)


def forward(input_vals:np.ndarray, net:NN):
    net.layers[0] = input_vals


    for index in range(1, len(net.layers), 1):
        shape = net.layers[index].activations.shape
        new_z_ly = np.zeros(ly_shape)
        new_ly   = np.zeros(ly_shape)
        #net.z_layers[index] = np.dot(net.layers[(index-1)], net.weights[index-1]) + net.biases[index-1]
        #todo: find a preforment way to do this!
        for i in shape[0]:
            for j in shape[1]:
                new_z_ly[i,j] = net.layers[(index - 1)].activations[i,j]* net.weights[index-1].weights[i,j]
                



#todo:
    #figure out network autocreation
    #figure out padding algorrithm
    #figure out down sizing

In [ ]:
config = NN_Config(
    kernel_shape = (5,5)
)

kernel_shape = (5,5)
l0_shape = (28,28)
l1_shape = (10,10)
l2_shape = (10,10)

net = NN.create_network(config, l0_shape, 3, l1_shape, 10)


In [ ]:
for i in range(len(net.layers)):
    print(net.layers[i].activations.shape)

for i in range(len(net.biases)):
    print(net.biases[i].biases.shape)

for i in range(len(net.weights)):
    print(net.weights[i].weights.shape)
